### Verb Tenses

In [53]:
import json


# Estrutura de tempos verbais
verb_tenses = {
    "present_continuous": {
        "structure": "{subject} is {verbing}"
    },
    "past_simple": {
        "structure": "{subject} {verbed}"
    },
    "present_simple": {
        "structure": "{subject} {verb}"
    }
}

# Regras de ortografia para adicionar terminações
def spelling_rules(word: str, termination: str) -> str:
    """
    Aplica regras de ortografia para adicionar uma terminação a uma palavra.

    Regras:
    1. Se termina com "ie", troca por "y" antes de adicionar a terminação.
    2. Se termina com "e" e a terminação começa com "ing", remove o "e".
    3. Se a palavra tem uma sílaba e termina em CVC (consoante-vogal-consoante),
       duplica a última consoante antes de adicionar a terminação.
    4. Caso contrário, apenas adiciona a terminação.
    """
    vowels = "aeiou"
    exceptions = "wxy"

    if word.endswith("ie"):
        word = word[:-2] + "y"
    elif word.endswith("e") and termination.startswith("ing"):
        word = word[:-1]
    elif (
        len(word) >= 3 and
        word[-1] not in vowels + exceptions and
        word[-2] in vowels and
        word[-3] not in vowels and
        termination.startswith("ing")
    ):
        word += word[-1]

    return word + termination

class VerbConjugator:
    def __init__(self):
        self.vowels = "aeiou"
        self.exceptions = "wxy"

        with open("data/irregular_verbs.json", "r", encoding="utf-8") as f:
            self.irregular_verbs = {entry["base"]: entry for entry in json.load(f)}


        self.verb_tenses = {
            "present_simple": {
                "structure": "{subject} {verb}",
                "requires_conjugation": True,
                "uses": ["verb"],
                "aux": None
            },
            "present_continuous": {
                "structure": "{subject} {aux} {verbing}",
                "requires_conjugation": False,
                "uses": ["verbing"],
                "aux": {
                    "type": "be",
                    "present": {
                        "i": "am",
                        "you": "are",
                        "we": "are",
                        "they": "are",
                        "he": "is",
                        "she": "is",
                        "it": "is"
                    }
                }
            },
            "past_simple": {
                "structure": "{subject} {verbed}",
                "requires_conjugation": False,
                "uses": ["verbed"],
                "aux": None
            },
            "present_perfect": {
                "structure": "{subject} {aux} {verbed}",
                "requires_conjugation": False,
                "uses": ["verbed"],
                "aux": {
                    "type": "have",
                    "present": {
                        "i": "have",
                        "you": "have",
                        "we": "have",
                        "they": "have",
                        "he": "has",
                        "she": "has",
                        "it": "has"
                    }
                }
            },
            "future_continuous": {
                "structure": "{subject} will be {verbing}",
                "requires_conjugation": False,
                "uses": ["verbing"],
                "aux": {
                    "type": "will_be",
                    "fixed": "will be"
                }
            }
        }

        # self.verb_tenses = {
        #     "present_simple": "{subject} {verb}",
        #     "present_continuous": "{subject} {aux} {verbing}",
        #     "past_simple": "{subject} {verbed}",
        #     "present_perfect": "{subject} have {verbed}",
        #     "future_continuous": "{subject} will be {verbing}",
        # }

        self.third_person_singular = {"he", "she", "it"}

    def spelling_rules(self, word: str, termination: str) -> str:
        if word.endswith("ie"):
            word = word[:-2] + "y"
        elif word.endswith("e") and termination.startswith("ing"):
            word = word[:-1]
        elif (
            len(word) >= 3 and
            word[-1] not in self.vowels + self.exceptions and
            word[-2] in self.vowels and
            word[-3] not in self.vowels and
            termination.startswith("ing")
        ):
            word += word[-1]
        return word + termination
    
    def conjugate_third_person(self, verb: str) -> str:
        if verb.endswith("y") and verb[-2] not in self.vowels:
            return verb[:-1] + "ies"
        elif verb.endswith(("o", "ch", "sh", "x", "s", "z")):
            return verb + "es"
        else:
            return verb + "s"

    def conjugate(self, verb: str, tense: str, subject: str = "I") -> str:
        tense_data = self.verb_tenses.get(tense)
        if not tense_data:
            return f"Tempo verbal '{tense}' não encontrado."

        structure = tense_data["structure"]
        uses = tense_data.get("uses", [])
        requires_conjugation = tense_data.get("requires_conjugation", False)


        # Gerúndio e passado (Formas verbais
        verbing = self.spelling_rules(verb, "ing")
        verbed = self.irregular_verbs.get(verb, {}).get("participle") or self.spelling_rules(verb, "ed")
        past = self.irregular_verbs.get(verb, {}).get("past") or self.spelling_rules(verb, "ed")

        # Conjugação para terceira pessoa
        if requires_conjugation and subject.lower() in self.third_person_singular:
            verb = self.conjugate_third_person(verb)

        # Auxiliar
        aux_data = tense_data.get("aux")
        if aux_data:
            if "fixed" in aux_data:
                aux = aux_data["fixed"]
            elif "present" in aux_data:
                aux = aux_data["present"].get(subject.lower(), aux_data["present"].get("they"))
            else:
                aux = ""
        else:
            aux = ""

        return structure.format(
            subject=subject,
            verb=verb,
            verbing=verbing,
            verbed=verbed,
            aux=aux
        )


conjugator = VerbConjugator()


test_cases = [
    ("do", "past_simple", "we"),
    ("go", "present_simple", "we"),
    ("go", "present_continuous", "we"),
    ("go", "past_simple", "we"),
    ("go", "present_perfect", "we"),
    ("go", "future_continuous", "she")
]

for verb, tense, subject in test_cases:
    print(conjugator.conjugate(verb=verb, tense=tense, subject=subject))

we done
we go
we are going
we gone
we have gone
she will be going


In [54]:
print(conjugator.conjugate("play", "present_simple", "He"))         # He plays
print(conjugator.conjugate("love", "present_continuous", "I"))      # I am loving
print(conjugator.conjugate("talk", "past_simple", "They"))          # They talked
print(conjugator.conjugate("work", "present_perfect", "We"))        # We have worked
print(conjugator.conjugate("jump", "future_continuous", "She"))     # She will be jumping

He plays
I am loving
They talked
We have worked
She will be jumping


In [56]:
import tkinter as tk
from tkinter import messagebox
# from verb_conjugator import VerbConjugator  # Sua classe deve estar nesse arquivo

class VerbTrainerApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Treinador de Conjugação 🇬🇧")
        self.root.geometry("500x350")
        self.conjugator = VerbConjugator()

        # Lista de exercícios
        self.exercises = [
            {"verb": "go", "subject": "She", "tense": "past_simple"},
            {"verb": "play", "subject": "He", "tense": "present_simple"},
            {"verb": "run", "subject": "We", "tense": "future_continuous"},
            {"verb": "love", "subject": "I", "tense": "present_perfect"},
            {"verb": "shop", "subject": "They", "tense": "present_continuous"},
            {"verb": "study", "subject": "He", "tense": "present_simple"},
            {"verb": "make", "subject": "I", "tense": "present_continuous"},
        ]
        self.current_index = 0

        # Interface
        self.exercise_label = tk.Label(root, text="", font=("Arial", 14))
        self.exercise_label.pack(pady=20)

        self.answer_entry = tk.Entry(root, font=("Arial", 14))
        self.answer_entry.pack(pady=10)

        self.feedback_label = tk.Label(root, text="", font=("Arial", 12), fg="red")
        self.feedback_label.pack(pady=5)

        self.progress_label = tk.Label(root, text="", font=("Arial", 10))
        self.progress_label.pack(pady=5)

        tk.Button(root, text="Verificar", command=self.check_answer).pack(pady=10)
        tk.Button(root, text="Reiniciar", command=self.restart).pack(pady=5)

        self.load_exercise()

    def load_exercise(self):
        if self.current_index >= len(self.exercises):
            self.exercise_label.config(text="🎉 Parabéns! Você concluiu todos os exercícios.")
            self.answer_entry.config(state="disabled")
            self.feedback_label.config(text="")
            self.progress_label.config(text=f"{len(self.exercises)} / {len(self.exercises)}")
            return

        ex = self.exercises[self.current_index]
        self.exercise_label.config(
            text=f"Conjugue: verbo='{ex['verb']}', sujeito='{ex['subject']}', tempo='{ex['tense']}'"
        )
        self.answer_entry.delete(0, tk.END)
        self.feedback_label.config(text="")
        self.progress_label.config(text=f"{self.current_index + 1} / {len(self.exercises)}")

    def check_answer(self):
        user_input = self.answer_entry.get().strip()
        ex = self.exercises[self.current_index]
        correct = self.conjugator.conjugate(ex["verb"], ex["tense"], ex["subject"])

        if user_input.lower() == correct.lower():
            self.feedback_label.config(text="✅ Correto!", fg="green")
            self.current_index += 1
            self.root.after(1000, self.load_exercise)
        else:
            self.feedback_label.config(text=f"❌ Errado. Correto seria: '{correct}'", fg="red")

    def restart(self):
        self.current_index = 0
        self.answer_entry.config(state="normal")
        self.load_exercise()

if __name__ == "__main__":
    root = tk.Tk()
    app = VerbTrainerApp(root)
    root.mainloop()

Exception in Tkinter callback
Traceback (most recent call last):
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.2800.0_x64__qbz5n2kfra8p0\Lib\tkinter\__init__.py", line 1968, in __call__
    return self.func(*args)
           ^^^^^^^^^^^^^^^^
  File "C:\Users\guilh\AppData\Local\Temp\ipykernel_45572\1268226084.py", line 60, in check_answer
    ex = self.exercises[self.current_index]
         ~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^
IndexError: list index out of range


In [48]:
def explain_tense(tense):
    data = verb_tenses.get(tense)
    if not data:
        return f"Tempo verbal '{tense}' não encontrado."

    info = f"""
Tempo: {tense.title()}
Exemplo: {data['example']}
Tradução: {data.get('translation', 'N/A')}
Uso: {data.get('usage', 'N/A')}
"""
    return info.strip()

info = explain_tense("past continuous")
print(info)

Tempo verbal 'past continuous' não encontrado.


- Adicionar suporte a negativas e perguntas (ex: Do you play?, He doesn’t play)
- Suporte a contrações (ex: I’m playing, She’s gone)
- Tradução automática para português ou outros idiomas
- Suporte a formas negativas e interrogativas


Com essa estrutura, você pode facilmente adicionar tempos como:
- past_continuous
- future_perfect
- conditional
- present_perfect_continuous
Basta seguir o mesmo padrão.
